spark session.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    array_contains, col, coalesce, concat_ws, expr, length, lit,
)

# Same parquet-committer override as notebook 02 — cluster default points at an EMR
# class whose JAR isn't on the classpath.
spark = SparkSession.builder \
    .appName('FB_API_topics') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

Master: yarn
Spark version: 3.5.0


26/05/17 07:30:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


paths.

In [2]:
V2_PATH = '/user/s3348393/main/preprocessing/v2/parquet'

load v2 and filter to the residual corpus. filter out the duplicates and only keep match_type is null, and in english. Ignore some commercial entities found while working through the problem up front. 

the body text etc are arrays - pull out the first non-null value from each array (probably the first, but just being careful)

some of the bodies are empty - add in the title and description to try and up the number of rows we can use

In [3]:
df = spark.read.parquet(V2_PATH)
print('All v2 rows:    ', df.count())


def first_non_empty(col_name):
    """First non-null, non-empty element of an array column. Returns null if none."""
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")


#rejects
COMMERCIAL_BYLINES = {
    'Access',                            # Indigenous Employment Australia — job listings
    'Streamotion Pty Ltd',               # Kayo / Binge sports + entertainment streaming
    'SBS Australia',                     # SBS On Demand streaming promotions
    'SBS Arabic24',                      # SBS language-stream marketing
    'SBS Mandarin中文普通话',            # SBS language-stream marketing
    'The Squiz',                         # paid news newsletter
    'Hair Cooki',                           #hair care ad
    "Shell"
}


corpus = df.filter(
        (col('ad_seq_no') == 1) &
        col('match_type').isNull() &
        # most of the language values are null - keep nulls and english
        (col('languages').isNull() | array_contains('languages', 'en')) &
        ~col('bylines').isin(list(COMMERCIAL_BYLINES))
    ) \
    .withColumn('body_text',  first_non_empty('creative_bodies')) \
    .withColumn('desc_text',  first_non_empty('creative_link_descs')) \
    .withColumn('title_text', first_non_empty('creative_link_titles')) \
    .withColumn('body',
        concat_ws(' ',
            coalesce(col('body_text'),  lit('')),
            coalesce(col('desc_text'),  lit('')),
            coalesce(col('title_text'), lit('')),
        )
    ) \
    .filter(length(col('body')) > 0) \
    .drop('body_text', 'desc_text', 'title_text')

print('Residual corpus:', corpus.count())
corpus.select('page_name', 'bylines', 'body').show(3, truncate=80)

All v2 rows:     5796491


Residual corpus: 94358
+----------------------------------+-----------------------------+--------------------------------------------------------------------------------+
|                         page_name|                      bylines|                                                                            body|
+----------------------------------+-----------------------------+--------------------------------------------------------------------------------+
|                    Thrive by Five|               Thrive By Five|Our early learning and childcare centres are under stress, and early childhoo...|
|Time for Change - Change Aged Care|         United Workers Union|Important Information for Aged Care workers. \n\nYour union has compiled a li...|
|     Australian Ethical Investment|Australian Ethical Investment|The sooner you switch to animal-friendly super, the sooner factory farming an...|
+----------------------------------+-----------------------------+-----------------------

add to the stop words some generic keywords that kept popping up. 

In [4]:
from pyspark.ml.feature import StopWordsRemover

stop_words = StopWordsRemover.loadDefaultStopWords('english') + [
    # URL / web junk that survives tokenisation
    'https', 'http', 'www', 'com', 'org', 'au', 'co', 'html',
    # contraction fragments surviving minTokenLength=2
    're', 've', 'll',
    # generic fillers (high frequency, low topic-discrimination value)
    'help', 'time', 'like', 'need', 'make', 'take', 'people',
    'year', 'years', 'today', 'also', 'will', 'can', 'get',
    'see', 'know', 'one', 'two', 'new', 'now', 'us',
    # generic CTA (kept short — don't strip 'petition', 'donate', 'sign', etc.
    # since those carry topic signal)
    'click', 'learn',
    #others
    'australia', 'australian', "2022"
]

print('Stop-words list size:', len(stop_words))

Stop-words list size: 218


preprog pipeline. 

regex tokenizer, 
stop word remover
count vectoriser. 

re-run the pipeline a few times to tune the count vectoriser.

In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, CountVectorizer

tokenizer = RegexTokenizer(
    inputCol='body', outputCol='raw_tokens',
    pattern=r'\W+', toLowercase=True, minTokenLength=2,
)

remover = StopWordsRemover(
    inputCol='raw_tokens', outputCol='tokens',
    stopWords=stop_words,
)

vectorizer = CountVectorizer(
    inputCol='tokens', outputCol='features',
    vocabSize=5000, #was 10000
    minDF=100,    # was 50
    maxDF=0.3,    # seems ok?
)

prep_pipeline = Pipeline(stages=[tokenizer, remover, vectorizer])

i did a bunch of runs, and apparently cache should help.

In [6]:
prep_model  = prep_pipeline.fit(corpus)
features_df = prep_model.transform(corpus).cache()

vocab = prep_model.stages[-1].vocabulary
print('Vocabulary size:', len(vocab))
print('\nTop 30 vocabulary terms (most frequent first):')
for i in range(0, 30, 3):
    print(f"{vocab[i]:<18}{vocab[i+1]:<18}{vocab[i+2]}")

Vocabulary size: 4251

Top 30 vocabulary terms (most frequent first):
sign              climate           government
support           community         petition
change            vote              world
protect           women             future
action            local             join
free              election          stop
woodside          share             council
children          life              donate
energy            every             labor
gas               tell              health


26/05/17 07:30:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


test some different group sizes on 10% data sample.

In [7]:
from pyspark.ml.clustering import LDA

#10%
sample_df = features_df.sample(0.1, seed=42).cache()

# Lookup from CountVectorizer integer term indices back to readable words.
vocab = prep_model.stages[-1].vocabulary

# Fit LDA at each k. seed=42 fixed so k=5 vs k=10 etc. are comparable.
for k in [5, 10, 15, 20, 25, 30]:
    print(k)
    lda = LDA(featuresCol='features', k=k, maxIter=20, seed=42)
    model = lda.fit(sample_df)
    topics = model.describeTopics(maxTermsPerTopic=10).collect()
    for row in topics:
        words = ' '.join(vocab[i] for i in row.termIndices)
        print(f'  Topic {row.topic:>2}: {words}')

sample_df.unpersist()

5


26/05/17 07:30:53 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: vote government community women council children support local election every
  Topic  1: sign petition government woodside enough stop support tell gas donate
  Topic  2: early solar learning support childcare refugees families day world educators
  Topic  3: climate sign action community change government support free petition share
  Topic  4: protect oceans woodside donate climate gas project ocean stop sign
10


26/05/17 07:31:05 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: product description name child super add join animal ethical queensland
  Topic  1: sign woodside petition oceans share tell ocean enough stop global
  Topic  2: solar human world survey watch interview ants breakthrough condition yellow
  Topic  3: election abc independent climate parliament sign party free women federal
  Topic  4: gas woodside climate donate protect project whales future fossil stop
  Topic  5: wrc forest forestry industry 50 climate mayoral communities products tasmanian
  Topic  6: support sign children donate plastic life women use free lives
  Topic  7: government labor school cost every morrison medicines education million support
  Topic  8: climate community government vote council action energy local change support
  Topic  9: children early women sign learning kids agl petition every childcare
15


...

26/05/17 07:31:36 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: child deforestation labour name add european 10 end globally experience
  Topic  1: weekly media issue companies urge oz arab care multinational fuelling
  Topic  2: november sure counted 2020 early learning americans tropics mt overseas
  Topic  3: wrc mayoral 50 1977 forensic communities hood heard candidate life
  Topic  4: super ethical join september 400 pds consider read information product
  Topic  5: forestry industry forest children yemen products war countries tasmanian timber
  Topic  6: donate refugees support provide ukraine donation families food refugee crisis
  Topic  7: level laws crossing crossings defence strong quoll environment gone spotted
  Topic  8: community climate council government action emissions business working future health
  Topic  9: early learning every childcare sign children families petition child kids
  Topic 10: native child wildlife name stand sex add protect logging sexual
  Topic 11: climate vote sign election women woodside enoug

26/05/17 07:31:47 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: victor local community council experience city including nominate services children
  Topic  1: issue media urge weekly companies oz arab multinational fuelling gave
  Topic  2: offshore emissions almost report carbon early learning greenhouse institute religious
  Topic  3: wrc mayoral 50 1977 forensic communities hood life bring back
  Topic  4: super ethical join september consider 400 pds information switch product
  Topic  5: forestry industry forest children war yemen tasmanian products tasmania countries
  Topic  6: support donate refugees ukraine provide food donation families crisis lives
  Topic  7: level laws defence crossings crossing strong quoll gone spotted environment
  Topic  8: climate action community change government native emissions crisis council victoria
  Topic  9: early learning childcare every families sign petition children affordable fair
  Topic 10: name child product vote stand sex add sexual protect wildlife
  Topic 11: sign women vote woodsi

DataFrame[id: string, page_id: string, page_name: string, snapshot_date: date, ad_creation_date: date, ad_delivery_start_date: date, ad_delivery_stop_date: date, creative_bodies: array<string>, creative_link_captions: array<string>, creative_link_descs: array<string>, creative_link_titles: array<string>, spend_lower_bound: bigint, spend_upper_bound: bigint, spend_mid: double, impressions_lower_bound: bigint, impressions_upper_bound: bigint, impressions_mid: double, audience_size_lower_bound: bigint, audience_size_upper_bound: bigint, audience_size_mid: double, currency: string, languages: array<string>, publisher_platforms: array<string>, demographic_distribution: array<struct<age:string,gender:string,percentage:string>>, delivery_by_region: array<struct<percentage:string,region:string>>, ad_snapshot_url: string, bylines: string, ad_seq_no: int, match_type: string, political_party: string, body: string, raw_tokens: array<string>, tokens: array<string>, features: vector]


below 20 blurs, above 20 some of the bags dont make sense. at 20 we can clearly see some topic clusters which dont look political, and can remove them.

In [ ]:
from pyspark.ml.clustering import LDA
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

K = 20

# Fit LDA at k=K on the full cached features. seed=42 for reproducibility.
lda = LDA(featuresCol='features', k=K, maxIter=20, seed=42)
lda_model = lda.fit(features_df)

# Transform: attach topicDistribution and dominant topic_id to every ad.
classified = lda_model.transform(features_df) \
    .withColumn('topic_array', vector_to_array('topicDistribution')) \
    .withColumn('topic_id', expr('array_position(topic_array, array_max(topic_array)) - 1'))

# Vocabulary for term-index -> word translation.
vocab = prep_model.stages[-1].vocabulary

# Top 15 terms per topic from describeTopics.
topics_rows = lda_model.describeTopics(maxTermsPerTopic=15).collect()
topic_terms = {row.topic: [vocab[i] for i in row.termIndices] for row in topics_rows}

# Top 5 bylines per topic — Spark window function over (topic_id, bylines, count).
w = Window.partitionBy('topic_id').orderBy(desc('count'))
top_bylines_rows = classified.filter(col('bylines').isNotNull()) \
    .groupBy('topic_id', 'bylines').count() \
    .withColumn('rank', row_number().over(w)) \
    .filter(col('rank') <= 5) \
    .orderBy('topic_id', 'rank') \
    .collect()

top_bylines = {}
for row in top_bylines_rows:
    top_bylines.setdefault(row.topic_id, []).append(row.bylines)

# Print to console — easy to scan while you draft labels.
print(f'k = {K}\n')
for tid in range(K):
    words = ' '.join(topic_terms.get(tid, []))
    bls   = ' | '.join(top_bylines.get(tid, []))
    print(f'Topic {tid:>2}:')
    print(f'  Terms:   {words}')
    print(f'  Bylines: {bls}')
    print()

26/05/17 07:32:00 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


k = 20

Topic  0:
  Terms:   super product switch feral join ethical native animals future species minutes performance description horses consider
  Bylines: Australian Ethical Investment | Invasive Species Council | Prime Super | 1in50 Incorporated | Future Super

Topic  1:
  Terms:   donate christmas provide give food ukraine support crisis donation emergency please care gift families water
  Bylines: Australia for UNHCR | Médecins Sans Frontières Australia | Wayside Chapel | Oxfam Australia | Invasive Species Council

Topic  2:
  Terms:   early learning childcare educators families support petition sign children quiz parents childhood affordable many accessible
  Bylines: Thrive By Five | Amnesty International Australia | TWU Members First Team | Danny Pearson MP for Essendon | Gender Awareness Australia

Topic  3:
  Terms:   government parliament nsw community mp state council public federal member news minister kids national armenian
  Bylines: Private Media | Amnesty Internationa

26/05/17 09:26:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_6 !
26/05/17 09:26:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_1 !
26/05/17 09:26:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_0 !
26/05/17 09:26:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_12 !
26/05/17 09:26:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_7 !
26/05/17 09:26:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_45_9 !
26/05/17 09:26:23 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 1 for reason Container marked as failed: container_1776819796568_1546_01_000004 on host: ip-100-64-73-193.ap-southeast-2.compute.internal. Exit status: -100. Diagnostics: Container released on a *lost* node.
26/05/17 09:26:23 ERROR YarnScheduler: Lost executor 1 on ip-100-64-73-193.ap-southeast-2.compute.internal: Container marked as f